# Does Modeling Competing Traders Improve Optimal Execution?

This notebook walks through the exact deterministic forecast game, its numerical audit, a causal stochastic policy and a finite-player deviation check. All examples are synthetic. The full [research report](../results/published/report.md) retains the 512-path study. See [methodology](../docs/methodology.md) for assumptions and derivations.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from meanfield_execution import MarketConfig, default_types, EquilibriumSystem, fixed_point, simulate
from meanfield_execution.population import population_audit
market = MarketConfig()
types = default_types()
print("Decision intervals:", market.steps)
print("Types:", [agent.name for agent in types])

Decision intervals: 40
Types: ['patient', 'balanced', 'urgent']


## 1. Compute and verify the forecast equilibrium

The mean-flow best-response map is affine. Compare the damped iteration with an independent linear solve. The residual is measured before damping; a small step alone is not evidence of consistency.

In [2]:
system = EquilibriumSystem(types, market.steps, market.dt, market.permanent_impact)
q0 = [agent.initial_inventory for agent in types]
forecast = np.linspace(0.08, 0, market.steps)
mean_flow, plans = system.solve(q0, forecast)
iteration = fixed_point(system, q0, forecast, damping=0.6)
print("Converged:", iteration.converged)
print("Undamped residual:", iteration.residuals[-1])
print("Max difference from direct solve:", np.max(np.abs(mean_flow - iteration.mean_rate)))
print("Damped iteration spectral radius:", system.spectral_radius(0.6))
assert np.allclose(mean_flow, types[0].weight * plans[0] + types[1].weight * plans[1] + types[2].weight * plans[2])

Converged: True
Undamped residual: 6.017504609128194e-11
Max difference from direct solve: 2.7227620158498667e-10
Damped iteration spectral radius: 0.41656271043067344


## 2. Expose false convergence from tiny damping

Tiny damping yields almost no movement but leaves a large undamped consistency error.

In [3]:
tiny = fixed_point(system, q0, forecast, damping=1e-12, max_iterations=3)
print("Converged:", tiny.converged)
print("Final consistency residual:", tiny.residuals[-1])
assert not tiny.converged

Converged: False
Final consistency residual: 6.282884250810762


## 3. Compare policies on the same exogenous path

Each decision uses information available before the next return. The filter removes known flow impact, updates the just-observed state, and predicts the next state. This is a receding-horizon conditional-mean approximation, not a solution of the full stochastic mean-field game.

In [4]:
records = []
for name in ["twap", "ac", "filtered_mfg", "naive_filter_mfg", "full_info_mfg"]:
    result = simulate(market, strategy=name, seed=20261117)
    records.append({"policy": name, "objective": result.metrics["objective"],
                    "terminal_inventory": result.metrics["terminal_inventory_rms"],
                    "accounting_error": result.metrics["accounting_error"]})
print(pd.DataFrame(records).round(8).to_string(index=False))
print("This is one path, not a performance estimate.")

          policy  objective  terminal_inventory  accounting_error
            twap   0.006337            0.000000               0.0
              ac   0.008194            0.016983               0.0
    filtered_mfg   0.016086            0.017450               0.0
naive_filter_mfg   0.020665            0.014677               0.0
   full_info_mfg   0.001636            0.024231               0.0
This is one path, not a performance estimate.


![Prespecified path](../results/published/figures/execution_path.png)

## 4. Check finite-player incentives

Every sampled agent uses its own initial inventory with the limiting flow forecast. The deviation test holds peers fixed and accounts for the deviator's own permanent impact. It covers open-loop deterministic schedules only.

In [5]:
audit = population_audit(market, types, population_size=100, seed=914)
for key, value in audit.items():
    print(f"{key}: {value}")
assert audit["max_deviation_gain"] >= audit["mean_deviation_gain"] >= 0

population_size: 100
seed: 914
mean_flow_rmse: 0.038950898736922666
terminal_impact_error: 0.003082265020521258
mean_deviation_gain: 3.058140954794283e-06
max_deviation_gain: 4.558650364653111e-06
sample_initial_inventory: 1.0515865166849863


## 5. Read the full experiment rather than select the best path

The main study shares each of 512 independent fundamental paths across seven policies. Confidence intervals are calculated across paths. The parameter grid is prespecified; failed convergence cases and losing policies remain in the output.

In [6]:
summary = pd.read_csv(ROOT / "results/published/summary.csv")
paired = summary[summary.metric == "objective_difference_vs_twap"]
print(paired[["strategy", "mean", "lower", "upper", "n"]].to_string(index=False))

        strategy      mean     lower     upper   n
            twap  0.000000  0.000000  0.000000 512
              ac -0.003278 -0.003989 -0.002567 512
    filtered_mfg -0.003091 -0.005118 -0.001065 512
   full_info_mfg -0.009579 -0.011665 -0.007493 512
naive_filter_mfg -0.001562 -0.003873  0.000750 512
misspecified_mfg -0.003152 -0.005219 -0.001084 512
       blind_mfg -0.000704 -0.002747  0.001340 512


![Finite populations](../results/published/figures/finite_population.png)

To reproduce the complete study from the repository root:

```bash
meanfield-execution --profile full --output results/reproduced
```

The equal-completion supplement below is implemented. Next experiments: add rate and inventory bounds, calibrate parameters on historical data, add transient impact with no-manipulation checks, and audit adaptive deviations. See the [limitations](../docs/limitations.md) before drawing empirical conclusions.

## Exact completion and actual decision information

The default study uses a soft terminal penalty. `hard_liquidation=True` instead imposes an exact KKT equality on every remaining-horizon plan. This does not impose nonnegative rates or intermediate inventory bounds. The auxiliary filter is still recorded for every strategy, but `decision_brier` follows actual planner information: zero when the current state is observed, the causal filter prior for filter-driven policies, and unavailable for policies that use no state probability.

In [7]:
from dataclasses import replace
hard_types = tuple(replace(agent, terminal_penalty=0, hard_liquidation=True) for agent in types)
hard_rows = []
for name in ["twap", "ac", "filtered_mfg", "full_info_mfg", "blind_mfg"]:
    hard = simulate(market, types=hard_types, strategy=name, seed=510000)
    assert np.allclose(hard.inventories[-1], 0, atol=1e-12)
    hard_rows.append({"strategy":name,
                      "terminal_inventory_rms":hard.metrics["terminal_inventory_rms"],
                      "decision_brier":hard.metrics["decision_brier"],
                      "auxiliary_prior_brier":hard.metrics["filter_prior_brier"]})
print(pd.DataFrame(hard_rows).to_string(index=False))
print("NaN means that the planner does not use a state probability; it is not a simulation failure.")

     strategy  terminal_inventory_rms  decision_brier  auxiliary_prior_brier
         twap            0.000000e+00             NaN               0.160571
           ac            3.103168e-18             NaN               0.160571
 filtered_mfg            5.898698e-19        0.160571               0.160571
full_info_mfg            5.485677e-19        0.000000               0.160571
    blind_mfg            0.000000e+00             NaN               0.160571


NaN means that the planner does not use a state probability; it is not a simulation failure.


## The equal-completion result

The separately labeled supplement retains 512 additional paired paths and all seven policies. The filtered policy's interval against TWAP spans zero. The simpler AC-style policy has the lower mean penalized objective, but the filtered-minus-AC interval also spans zero. These are objective comparisons conditional on the simulator, not a claim about live execution or unconditional shortfall superiority.

In [8]:
hard_summary = pd.read_csv(ROOT / "results/hard_liquidation/summary.csv")
comparisons = hard_summary[(hard_summary.strategy == "filtered_mfg") &
                          hard_summary.metric.isin(["objective_difference_vs_twap", "objective_difference_vs_ac"])]
print(comparisons[["strategy","metric","mean","lower","upper","n"]].to_string(index=False))
assert ((comparisons.lower < 0) & (comparisons.upper > 0)).all()
print("Both paired intervals span zero; preserve this result when discussing the project.")

    strategy                       metric      mean     lower    upper   n
filtered_mfg objective_difference_vs_twap -0.001241 -0.003270 0.000788 512
filtered_mfg   objective_difference_vs_ac  0.001190 -0.000126 0.002505 512
Both paired intervals span zero; preserve this result when discussing the project.
